# [실습 1] 커스텀 모델 헤드와 멀티태스크/앙상블 구성


## 실습 개요

> "사전학습 모델 위에 태스크별 출력 구조를 직접 붙이면 어떤 모델을 만들 수 있을까?"

이번 실습에서는 Transformers 백본 모델 위에 분류 헤드와 회귀 헤드를 직접 구성합니다.  
강의에서 다룬 커스텀 모델 헤드, 앙상블, 멀티태스크 구조를 작은 코드 예제로 확인합니다.

실습은 빠른 실행을 위해 작은 BERT 테스트 모델을 사용합니다. 핵심은 모델 품질이 아니라 구조와 forward 출력의 설계입니다.


## 실습 목표

1. `AutoModel`로 백본 모델을 로드한다.
2. CLS 벡터 위에 커스텀 분류 헤드를 붙인다.
3. 여러 헤드 출력을 동시에 반환하는 멀티태스크 모델을 만든다.
4. 여러 모델의 logits를 평균내는 간단한 앙상블을 구현한다.
5. [TODO] 분류/회귀 헤드가 있는 멀티태스크 모델을 완성한다.


## 데이터 출처

- 모델: `hf-internal-testing/tiny-random-bert` (Hugging Face Hub)
- 데이터: 실습용 문장 직접 구성


## 목차

1. 라이브러리 임포트 및 백본 모델 로드
2. 입력 토크나이징
3. 커스텀 분류 헤드 구현
4. 간단한 logits 앙상블
5. [TODO] 멀티태스크 모델 구현


---
## 1. 라이브러리 임포트 및 백본 모델 로드


### 백본 모델과 공통 라이브러리 준비

커스텀 헤드를 붙일 때는 PyTorch의 `nn.Module`과 Hugging Face의 `AutoConfig`, `AutoTokenizer`, `AutoModel`이 함께 사용됩니다. 토크나이저, 설정 객체, 백본 모델은 같은 checkpoint에서 가져와야 입력 형식과 hidden size가 서로 맞습니다.

처음에 `hidden_size`와 `model_type`을 확인해 두면 뒤에서 Linear 헤드의 입력 차원을 정할 때 기준이 생깁니다. 모델을 직접 감싸는 실습에서는 이런 기본 정보를 먼저 확인하는 습관이 디버깅 시간을 많이 줄여 줍니다.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from transformers import AutoConfig, AutoTokenizer, AutoModel

checkpoint = 'hf-internal-testing/tiny-random-bert'
config = AutoConfig.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
backbone = AutoModel.from_pretrained(checkpoint)
backbone.eval()

print('hidden_size:', config.hidden_size)
print('model_type:', config.model_type)

Loading weights: 100%|██████████| 87/87 [00:00<00:00, 12824.36it/s]
[transformers] BertModel LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
qa_outputs.bias                            | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
classifier.bias                            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
classifier.weight                          | U

hidden_size: 32
model_type: bert


---
## 2. 입력 토크나이징


### 텍스트를 모델 입력 텐서로 변환하기

자연어 문장은 모델에 바로 들어갈 수 없기 때문에 토크나이저를 거쳐 `input_ids`와 `attention_mask`로 바뀝니다. `padding=True`는 길이가 다른 문장을 한 배치로 묶기 위한 설정이고, `truncation=True`는 모델이 처리할 수 있는 길이를 넘는 입력을 잘라 냅니다.

여기서는 입력 텐서의 shape을 확인하는 데 집중합니다. 커스텀 헤드나 멀티태스크 구조를 만들더라도 입력 배치가 올바르게 만들어졌는지가 먼저 보장되어야 합니다.

In [9]:
texts = [
    'The service response was fast and helpful.',
    'The user experience was confusing.',
    'The model prediction should be monitored.',
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
for key, value in inputs.items():
    print(key, tuple(value.shape))

input_ids (3, 38)
token_type_ids (3, 38)
attention_mask (3, 38)


---
## 3. 커스텀 분류 헤드 구현


### 백본 표현 위에 분류 헤드 얹기

사전학습 백본은 문장을 문맥 벡터로 바꾸는 역할을 맡고, 태스크별 헤드는 그 벡터를 원하는 출력 형태로 변환합니다. 여기서는 첫 번째 토큰 위치의 벡터를 문장 대표 표현처럼 사용한 뒤 Dropout과 Linear 레이어를 거쳐 분류 logits를 만듭니다.

이 구조는 고객 문의 분류, 문서 카테고리 분류, 위험도 등급 예측처럼 라벨 개수가 정해진 문제에서 자주 쓰입니다. 백본을 바꾸더라도 `hidden_size`와 `num_labels`만 맞추면 같은 패턴을 유지할 수 있다는 점이 중요합니다.

`forward()`에서 딕셔너리를 반환하는 방식도 눈여겨볼 부분입니다. 출력 이름을 명확히 정해 두면 뒤에서 손실 함수, 평가 코드, Trainer 연동 코드를 붙일 때 어떤 값이 어디에 쓰이는지 추적하기 쉽습니다.

BERT 계열 토크나이저는 문장 쌍 구분을 위한 `token_type_ids`를 함께 만들 수 있습니다. 입력 배치를 `classifier(**inputs)`처럼 그대로 넘길 수 있도록 forward 인자에 이 값을 포함해 두면 모델 호출부가 더 안정적입니다.

In [4]:
class CustomClassifier(nn.Module):
    def __init__(self, checkpoint, num_labels=3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_vector = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(cls_vector))
        return logits

classifier = CustomClassifier(checkpoint, num_labels=3)
classifier.eval()

with torch.no_grad():
    logits = classifier(**inputs)

print('logits shape:', tuple(logits.shape))
print(logits)

Loading weights: 100%|██████████| 87/87 [00:00<00:00, 9819.55it/s]
[transformers] BertModel LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
qa_outputs.bias                            | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
classifier.bias                            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
classifier.weight                          | UN

logits shape: (3, 3)
tensor([[-0.0802, -0.1967,  0.9172],
        [-0.0780, -0.1963,  0.9179],
        [-0.0798, -0.1985,  0.9163]])


---
## 4. 간단한 logits 앙상블


### 여러 모델의 예측을 결합하는 앙상블

같은 입력에 대해 두 모델의 logits를 계산한 뒤 평균을 내면 가장 단순한 형태의 앙상블이 됩니다. 실제 프로젝트에서는 서로 다른 seed, 데이터 분할, 모델 구조를 조합해 단일 모델이 가진 우연한 편향을 줄일 때 사용합니다.

logits를 평균낸 뒤 softmax를 적용하는 방식과 각 모델의 확률을 평균내는 방식은 결과가 달라질 수 있습니다. 이번 예제는 구조를 보기 위한 최소 구현이므로 평균만 사용하지만, 실험에서는 모델별 가중치나 검증 성능을 반영한 결합 방식도 함께 비교합니다.

In [5]:
model_a = CustomClassifier(checkpoint, num_labels=3)
model_b = CustomClassifier(checkpoint, num_labels=3)
model_a.eval()
model_b.eval()

with torch.no_grad():
    logits_a = model_a(**inputs)
    logits_b = model_b(**inputs)
    ensemble_logits = (logits_a + logits_b) / 2
    ensemble_probs = torch.softmax(ensemble_logits, dim=-1)

print('ensemble_probs shape:', tuple(ensemble_probs.shape))
print('예측 클래스:', torch.argmax(ensemble_probs, dim=-1).tolist())

Loading weights: 100%|██████████| 87/87 [00:00<00:00, 12634.32it/s]
[transformers] BertModel LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
qa_outputs.bias                            | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
classifier.bias                            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
classifier.weight                          | U

ensemble_probs shape: (3, 3)
예측 클래스: [2, 2, 2]


---
### [TODO] 멀티태스크 모델 구현

공유 백본 위에 두 개의 헤드를 가진 `MultiTaskModel`을 완성하세요.

구현 조건:
- `self.backbone`은 `AutoModel.from_pretrained(checkpoint)`로 만듭니다.
- `self.classification_head`는 `num_labels`개 출력을 만듭니다.
- `self.regression_head`는 1개 점수를 출력합니다.
- `forward()`는 `classification_logits`와 `regression_score`를 딕셔너리로 반환합니다.


### 공유 백본에서 여러 태스크 출력 만들기

멀티태스크 모델은 하나의 백본이 공통 표현을 만들고, 그 위에 태스크별 헤드를 따로 두는 구조입니다. 분류 헤드는 클래스별 logits를 만들고, 회귀 헤드는 입력마다 하나의 연속값 점수를 만듭니다.

리뷰 텍스트에서 감성 등급과 만족도 점수를 동시에 예측하거나, 상담 문장에서 문의 유형과 긴급도를 함께 예측하는 상황에 적용할 수 있습니다. 여러 태스크가 비슷한 언어 표현을 공유할수록 백본 파라미터를 더 효율적으로 활용할 수 있습니다.

다만 모든 태스크를 무조건 함께 학습하는 것이 좋은 것은 아닙니다. 한 태스크의 손실이 지나치게 크거나 데이터 규모가 많이 다르면 다른 태스크의 학습을 방해할 수 있으므로, 손실 가중치와 배치 구성까지 함께 설계해야 합니다.

앞에서 만든 입력 배치를 그대로 재사용하므로 멀티태스크 모델도 `token_type_ids`를 받을 수 있어야 합니다. 이렇게 맞춰 두면 모델 구조를 바꿔도 토크나이저 출력과 forward 인자 사이의 불일치가 줄어듭니다.

In [12]:
class MultiTaskModel(nn.Module):
    def __init__(self, checkpoint, num_labels=3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(checkpoint)
        hidden_size = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classification_head = nn.Linear(hidden_size, num_labels)
        self.regression_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        cls_vector = outputs.last_hidden_state[:, 0]
        cls_vector = self.dropout(cls_vector)
        classification_logits = self.classification_head(cls_vector)
        regression_score = self.regression_head(cls_vector)
        return {
            'classification_logits': classification_logits,
            'regression_score': regression_score,
        }

### 멀티태스크 출력 형태 점검

커스텀 모델을 학습하기 전에는 작은 배치 하나로 forward 결과를 확인하는 편이 좋습니다. `classification_logits`는 `(batch_size, num_labels)` 형태여야 하고, `regression_score`는 입력마다 하나의 점수를 반환해야 합니다.

shape이 맞지 않으면 손실 함수 계산이나 Trainer 연동 단계에서 오류가 뒤늦게 발생합니다. 구조를 바꾼 직후에 출력 키와 차원을 확인해 두면 문제 원인을 훨씬 빨리 좁힐 수 있습니다.

In [13]:
multi_model = MultiTaskModel(checkpoint, num_labels=3)
multi_model.eval()

with torch.no_grad():
    multitask_outputs = multi_model(**inputs)

print('classification:', tuple(multitask_outputs['classification_logits'].shape))
print('regression:', tuple(multitask_outputs['regression_score'].shape))

Loading weights: 100%|██████████| 87/87 [00:00<00:00, 10308.04it/s]
[transformers] BertModel LOAD REPORT from: hf-internal-testing/tiny-random-bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
qa_outputs.bias                            | UNEXPECTED |  | 
qa_outputs.weight                          | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
classifier.bias                            | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
classifier.weight                          | U

classification: (3, 3)
regression: (3, 1)
